# Real-model validation: two-hop factual recall in GPT-2
Runtime → **T4 GPU** (about 1–1.5 h). Trains a PLT and a CLT for GPT-2 small at an identical budget on WikiText-103, then runs four tests on two-hop prompts (*the capital of the state containing Dallas*) against one-hop controls (*the capital of Texas*):
* **V1** depth sensitivity: D_feat(2-hop) − D_feat(1-hop)
* **V2** causal specificity: state features vs random features (leave-one-out selection)
* **V3** graph claim vs intervention
* **V4** CLT vs PLT on the same prompts

The default arguments reproduce the reported run.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import glob, os, sys
zips = glob.glob('/content/clt-depth-collapse*.zip')
if zips:                                   # option A: uploaded zip of this repository
    !unzip -q -o "{zips[0]}" -d /content/
else:                                      # option B: clone from GitHub
    !git clone -q https://github.com/sirikrishna-dev/clt-depth-collapse.git /content/clt-depth-collapse
os.chdir('/content/clt-depth-collapse')
!pip -q install -r requirements.txt transformers datasets
!python -m pytest -q tests

In [ ]:
OUT = '/content/drive/MyDrive/clt_results/gpt2_twohop'
!PYTHONPATH=src python -m cdc.realdata.lm_twohop --model gpt2 --out $OUT

In [ ]:
import pandas as pd
p = pd.read_csv(f'{OUT}/prompts.csv'); print(p.groupby('hop').kept.sum())
print(pd.read_csv(f'{OUT}/lm_fidelity.csv')[['kind', 'n_params', 'heldout_nmse_mean']])
print(pd.read_csv(f'{OUT}/lm_validation_tests.csv').to_string(index=False))
!python scripts/make_lm_tables.py --results $OUT --paper paper/generated